In [ ]:
# Setup: credentials from .env (never hardcode)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from notebooks.load_env import load_env, get_project_root
load_env()
PROJECT_ROOT = get_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "archive"

In [ ]:
# !uv sync

# 01 — Prepare Postgres Data (Vector Store)

News chunks + embeddings for PGVector. **Prerequisites:** `.env` configured, `docker compose up -d`.

---

## 1. Load and Clean Data

In [ ]:
content_path = DATA_DIR / "data.csv"
rating_path = DATA_DIR / "rating.csv"

In [ ]:
import pandas as pd

# 1. Load the data
df_content = pd.read_csv(content_path)
df_rating = pd.read_csv(rating_path)

# 2. Drop rows missing 'full_content'
df_clean = df_content.dropna(subset=['full_content']).copy()

# 3. Merge with ratings to get sentiment labels
# Note: how='inner' ensures we only keep articles that have both text AND a sentiment label
final_df = df_clean.merge(
    df_rating[['article_id', 'title_sentiment']], 
    on='article_id', 
    how='inner'
)

# 4. Final Cleanup: Select only necessary columns to keep Postgres lean
columns_to_keep = [
    'article_id', 'source_name', 'author', 'title', 
    'published_at', 'full_content', 'category', 'title_sentiment'
]
final_df = final_df[columns_to_keep]

print(f"Original size: {len(df_content)}")
print(f"Cleaned size: {len(final_df)}")

#### Sampling 5k pilot - by published time and by sentiment distribution

In [ ]:
# 1. Ensure published_at is a datetime object (coerce invalid → NaT, then drop)
final_df['published_at'] = pd.to_datetime(final_df['published_at'], errors='coerce')
final_df = final_df.dropna(subset=['published_at'])

# 2. Stratified Sampling by Sentiment
# This takes an equal number of rows for each sentiment type
pilot_df = final_df.groupby('title_sentiment', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), 1666), random_state=42)
)

# 3. Sort by date to maintain chronological integrity for the agent
pilot_df = pilot_df.sort_values('published_at').reset_index(drop=True)

print(f"Pilot size: {len(pilot_df)}")
print(pilot_df['title_sentiment'].value_counts())

In [ ]:
pilot_df

## Create Embeddings for the pilot data: full_content + title

In [ ]:
# Word count stats for full_content (for chunking decisions) — BEFORE truncation
word_counts = pilot_df['full_content'].str.split().str.len()

print("=== full_content: Word count descriptive stats (pre-truncation) ===\n")
print(f"{'Count':<14} {len(word_counts):,}")
print(f"{'Min':<14} {word_counts.min():,.0f}")
print(f"{'Max':<14} {word_counts.max():,.0f}")
print(f"{'Mean':<14} {word_counts.mean():,.1f}")
print(f"{'Median':<14} {word_counts.median():,.0f}")
print(f"{'Std':<14} {word_counts.std():,.1f}")
print("\nPercentiles:")
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  P{p:<3}  {word_counts.quantile(p/100):,.0f}")

### Truncate full_content to max 3k words <P95

In [ ]:
# Truncate full_content to max 5k words
MAX_WORDS = 3000

def truncate_to_words(text, max_words=MAX_WORDS):
    if pd.isna(text):
        return text
    words = str(text).split()
    return " ".join(words[:max_words]) if len(words) > max_words else text

pilot_df['full_content'] = pilot_df['full_content'].apply(truncate_to_words)

truncated = pilot_df['full_content'].str.split().str.len()
print(f"Truncated to max {MAX_WORDS:,} words:")
print(f"  Max now: {truncated.max():,.0f} | Articles at limit: {(truncated >= MAX_WORDS).sum():,}")

In [ ]:
# Minimally Clean text
import re

def clean_for_embedding(text):
    # 1. Remove URLs and Emails
    text = re.sub(r'\S*@\S*\s?', '', text) 
    text = re.sub(r'http\S+', '', text)
    
    # 2. Remove common news "Noise" phrases (Customize this list)
    noise_phrases = ["Click here to read more", "Follow us on social media", "Advertisement"]
    for phrase in noise_phrases:
        text = text.replace(phrase, "")
        
    # 3. Standardize whitespace
    text = " ".join(text.split())
    return text

# Apply to your Pilot DF
pilot_df['full_content'] = pilot_df['full_content'].apply(clean_for_embedding)

In [ ]:
# CHUCKING HERE 

# Set chunking params for pilot
CHUNK_SIZE = 1000   # chars (~250 tokens; fits <256 context)
CHUNK_OVERLAP = 100 # chars (~25 tokens); small, minimizes redundancy

def fast_safe_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):

    if pd.isna(text) or not text.strip():
        return []

    chunks = []
    text = text.strip()
    n = len(text)
    start = 0

    while start < n:

        end = start + chunk_size

        if end >= n:
            chunks.append(text[start:].strip())
            break

        chunk_end = text.rfind(' ', start, end)

        if chunk_end == -1 or chunk_end <= start:
            actual_end = end
        else:
            actual_end = chunk_end

        chunks.append(text[start:actual_end].strip())

        # Move start with overlap
        start = max(actual_end - overlap, 0)

        # Snap start to word boundary
        next_space = text.find(' ', start)
        if next_space != -1 and next_space < actual_end:
            start = next_space + 1

    return [c for c in chunks if len(c) > 10]

# Apply chunking to the pilot data
pilot_df['content_chunks'] = pilot_df['full_content'].apply(fast_safe_chunks)

#### Create embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
from tqdm.auto import tqdm
import time

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

# Explode chunks
chunks_df = (
    pilot_df[['content_chunks']]
    .explode('content_chunks', ignore_index=True)
    .rename(columns={'content_chunks': 'chunk_text'})
)

texts = chunks_df["chunk_text"].tolist()

print(f"Embedding {len(texts)} chunks on {device}...")

batch_size = 128
embeddings = []

start_time = time.time()

for i in tqdm(range(0, len(texts), batch_size), desc="Embedding progress"):
    
    batch = texts[i:i + batch_size]

    batch_emb = model.encode(
        batch,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    embeddings.append(batch_emb)

embeddings = np.vstack(embeddings)

elapsed = time.time() - start_time
print(f"\nEmbedding completed in {elapsed:.2f} seconds")

chunks_df["embedding"] = list(embeddings)

In [ ]:
# Upsert to Postgres — credentials from .env
import os
import pandas as pd
import psycopg2

df = pd.read_parquet(PROJECT_ROOT / "notebooks" / "pilot_chunks_with_embeddings.parquet", engine="fastparquet")

conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=int(os.getenv("POSTGRES_PORT", "5433")),
    database=os.getenv("POSTGRES_DB", "market_intel"),
    user=os.getenv("POSTGRES_USER", "admin"),
    password=os.getenv("POSTGRES_PASSWORD"),
)

cur = conn.cursor()
print("Connected to Postgres")

In [ ]:
pilot_chunks = (
    pilot_df
    .explode("content_chunks", ignore_index=True)
    .rename(columns={"content_chunks": "chunk_text"})
)


In [ ]:
final_df = pilot_chunks.merge(
    chunks_df[["chunk_text", "embedding"]],
    on="chunk_text",
    how="left"
)

In [ ]:
final_df.to_csv("pilot_chunks_with_embeddings.csv", index=False)

In [ ]:
final_df["embedding"] = final_df["embedding"].apply(
    lambda x: x.tolist() if isinstance(x, np.ndarray) else x
)

final_df.to_parquet(
    "pilot_chunks_with_embeddings.parquet",
    engine="fastparquet",
    compression="snappy",
    index=False
)

print("Saved successfully.")

In [ ]:
import pandas as pd
df = pd.read_parquet(
    "pilot_chunks_with_embeddings.parquet",
    engine="fastparquet"
)

df

### Push data to pgvector store 

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
import pandas as pd
import numpy as np

# 1. Load dataset
df = pd.read_parquet("pilot_chunks_with_embeddings.parquet", engine="fastparquet")

# 2. Database Connection
conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"), port=int(os.getenv("POSTGRES_PORT", "5433")), database=os.getenv("POSTGRES_DB", "market_intel"), user=os.getenv("POSTGRES_USER", "admin"), password=os.getenv("POSTGRES_PASSWORD")
)
cur = conn.cursor()

# 3. Schema Setup - FORCE REFRESH
print("Cleaning old schema and preparing new table...")
cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
# We drop the table because IF NOT EXISTS won't update columns if the table already exists
cur.execute("DROP TABLE IF EXISTS article_chunks CASCADE;") 

cur.execute("""
CREATE TABLE article_chunks (
    article_id TEXT NOT NULL,
    source_name TEXT,
    author TEXT,
    title TEXT,
    published_at TIMESTAMPTZ,
    category TEXT,
    title_sentiment TEXT,
    content TEXT NOT NULL,
    embedding vector(384),
    fts_tokens tsvector,
    chunk_hash TEXT GENERATED ALWAYS AS (md5(article_id || '::' || content)) STORED,
    PRIMARY KEY (chunk_hash)
)
""")
conn.commit()

# 4. Pre-processing
def to_list(v):
    if isinstance(v, np.ndarray): return v.tolist()
    return v

df["embedding"] = df["embedding"].apply(to_list)
df = df.drop_duplicates(subset=["article_id", "chunk_text"], keep="first")

# 5. Data Mapping
records = [
    (
        str(r.article_id),
        r.source_name,
        r.author,
        r.title,
        r.published_at if pd.notnull(r.published_at) else None,
        r.category,
        r.title_sentiment,
        r.chunk_text, 
        r.embedding
    )
    for r in df.itertuples(index=False)
]

# 6. Upsert Logic
sql = """
INSERT INTO article_chunks (
    article_id, source_name, author, title, published_at, 
    category, title_sentiment, content, embedding
)
VALUES %s
ON CONFLICT (chunk_hash) DO UPDATE SET
    source_name = EXCLUDED.source_name,
    author = EXCLUDED.author,
    title = EXCLUDED.title,
    published_at = EXCLUDED.published_at,
    category = EXCLUDED.category,
    title_sentiment = EXCLUDED.title_sentiment,
    embedding = EXCLUDED.embedding
"""

print(f"Ingesting {len(records)} records...")
execute_values(cur, sql, records, page_size=1000)
conn.commit()

# 7. Post-Ingest: Generate FTS Tokens
print("Updating Full-Text Search tokens...")
cur.execute("""
    UPDATE article_chunks 
    SET fts_tokens = to_tsvector('english', coalesce(title, '') || ' ' || coalesce(content, ''));
""")

# 8. Create Indices for Performance
print("Creating HNSW index...")
cur.execute("CREATE INDEX ON article_chunks USING hnsw (embedding vector_cosine_ops);")
cur.execute("CREATE INDEX ON article_chunks USING gin(fts_tokens);")

conn.commit()
print("Done! Database is fully operational.")

cur.close()
conn.close()

### Create pgvector IVFFlat index (run after upsert)

**Why this is crucial:** Vector search computes similarity `distance(query_embedding, stored_embedding)`.

- **Without index:** 27k vectors → every query checks all 27k rows
- **With IVFFlat:** vectors grouped into clusters → query searches only nearest clusters (e.g. ~278 per cluster with `lists=100` → ~300 rows vs 27k)

In [ ]:
# Create pgvector IVFFlat index for fast similarity search
import psycopg2

conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=int(os.getenv("POSTGRES_PORT", "5433")),
    database=os.getenv("POSTGRES_DB", "market_intel"),
    user=os.getenv("POSTGRES_USER", "admin"),
    password=os.getenv("POSTGRES_PASSWORD"),
)
cur = conn.cursor()

cur.execute("""
CREATE INDEX IF NOT EXISTS article_chunks_embedding_idx
ON article_chunks
USING ivfflat (embedding vector_cosine_ops)
WITH (lists = 100);
""")
conn.commit()
print("✓ IVFFlat index created")

cur.execute("ANALYZE article_chunks")
conn.commit()
print("✓ ANALYZE complete")

cur.close()
conn.close()

### Hybrid Search (BM25 + Vector) and HNSW Index

**1. BM25 (Full-Text Search):** Dense vectors are great for semantics but weak on exact names (e.g. "iOttie", "RationalStat"). BM25 (sparse, keyword) fixes this.

**2. HNSW Index:** At 5k+ rows, HNSW is the 2026 standard—handles incremental updates (new news daily) without degrading recall. IVFFlat needs periodic rebuilds; HNSW does not.

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=int(os.getenv("POSTGRES_PORT", "5433")),
    database=os.getenv("POSTGRES_DB", "market_intel"),
    user=os.getenv("POSTGRES_USER", "admin"),
    password=os.getenv("POSTGRES_PASSWORD"),
)
cur = conn.cursor()

# 1. Add TSVector column for BM25 (keyword) search
cur.execute("""
    ALTER TABLE article_chunks
    ADD COLUMN IF NOT EXISTS fts_tokens tsvector
    GENERATED ALWAYS AS (to_tsvector('english', content)) STORED;
""")
conn.commit()
print("✓ fts_tokens column added")

# 2. GIN index for FTS speed
cur.execute("""
    CREATE INDEX IF NOT EXISTS idx_fts ON article_chunks USING GIN (fts_tokens);
""")
conn.commit()
print("✓ GIN index on fts_tokens created")

# 3. HNSW index (graph-based, better for incremental updates)
cur.execute("""
    CREATE INDEX IF NOT EXISTS idx_hnsw_embedding ON article_chunks
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);
""")
conn.commit()
print("✓ HNSW index created")

cur.execute("ANALYZE article_chunks")
conn.commit()
print("✓ ANALYZE complete")

cur.close()
conn.close()

### RRF Hybrid Smoke Test (BM25 + Vector)

**Reciprocal Rank Fusion (RRF)** merges results from multiple rankers (keyword + semantic) without score calibration. Each list contributes `1/(k + rank)` per doc; we sum across lists. The constant `k=60` dampens top-heavy effects. **Scores are not 0–1 or 1–100** — typical RRF scores are small (e.g. 0.01–0.05). Higher = better, but compare *relative* rank, not absolute.

| RRF Score | Interpretation | Action |
|-----------|-----------------|--------|
| > 0.030 | Golden Match | Hit #1 in both Keyword and Vector. Extremely relevant. |
| 0.015–0.025 | Strong Match | Likely the top hit in one search engine. Very reliable. |
| 0.010–0.015 | Relevant | Top 10–20 hit. Good for context, but not a "direct" answer. |
| < 0.010 | Noise | Likely a low-ranking filler result. Ignore for final LLM synthesis. |

- **Keyword-heavy:** "iOttie" — BM25 excels at exact brand names
- **Semantic-heavy:** "market trends in EV charging" — vectors capture concepts

In [ ]:
import psycopg2
from sentence_transformers import SentenceTransformer

# 1. Setup
model = SentenceTransformer('all-MiniLM-L6-v2')
conn = psycopg2.connect(host=os.getenv("POSTGRES_HOST", "localhost"), port=int(os.getenv("POSTGRES_PORT", "5433")), database=os.getenv("POSTGRES_DB", "market_intel"), user=os.getenv("POSTGRES_USER", "admin"), password=os.getenv("POSTGRES_PASSWORD"))
cur = conn.cursor()

def smoke_test_hybrid(query_text):
    print(f"\n--- Testing Query: '{query_text}' ---")
    
    # Generate Embedding
    query_vector = model.encode(query_text).tolist()

    # RRF Hybrid Query
    # k=60 is the standard constant that balances top-ranked results
    hybrid_query = """
    WITH semantic_search AS (
        SELECT chunk_hash, ROW_NUMBER() OVER (ORDER BY embedding <=> %s::vector) as rank
        FROM article_chunks
        ORDER BY embedding <=> %s::vector
        LIMIT 20
    ),
    keyword_search AS (
        SELECT chunk_hash, ROW_NUMBER() OVER (ORDER BY ts_rank_cd(fts_tokens, plainto_tsquery('english', %s)) DESC) as rank
        FROM article_chunks
        WHERE fts_tokens @@ plainto_tsquery('english', %s)
        ORDER BY rank DESC
        LIMIT 20
    )
    SELECT 
        a.title,
        a.category,
        a.content,
        COALESCE(1.0 / (60 + s.rank), 0.0) + COALESCE(1.0 / (60 + k.rank), 0.0) AS rrf_score
    FROM semantic_search s
    FULL OUTER JOIN keyword_search k ON s.chunk_hash = k.chunk_hash
    JOIN article_chunks a ON a.chunk_hash = COALESCE(s.chunk_hash, k.chunk_hash)
    ORDER BY rrf_score DESC
    LIMIT 3;
    """

    cur.execute(hybrid_query, (query_vector, query_vector, query_text, query_text))
    results = cur.fetchall()

    if not results:
        print("No results found. Check if FTS tokens were generated.")
        return

    for i, (title, cat, content, score) in enumerate(results):
        print(f"{i+1}. [Score: {score:.4f}] [{cat}] {title}")
        print(f"   Snippet: {content[:150]}...\n")

# Run two distinct tests
smoke_test_hybrid("iOttie wireless charger price and safety")  # Keyword-heavy
smoke_test_hybrid("market trends in electronic vehicle charging")  # Semantic-heavy

cur.close()
conn.close()

In [ ]:
#HERE